# 07 · General capabilities: tinyBenchmarks

Compares the base model and the intervened model on tinyMMLU, tinyHellaswag, tinyARC, tinyTruthfulQA, tinyWinogrande (log-likelihood scoring of answer options) and tinyGSM8k (greedy generation). Reports raw accuracy and the IRT-based estimates from the `tinyBenchmarks` package.

In [ ]:
# !pip install -q -U datasets transformers accelerate pandas tqdm huggingface_hub safetensors
# !pip install -q git+https://github.com/felipemaiapolo/tinyBenchmarks

from google.colab import drive, userdata
from huggingface_hub import login

drive.mount('/content/drive')
HF_TOKEN = userdata.get('HF_TOKEN')
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    login()

from pathlib import Path
from contextlib import contextmanager, nullcontext
import json
import random
import re

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
import tinyBenchmarks as tb

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'meta-llama/Meta-Llama-3-8B-Instruct'
SAE_OUT_DIR = Path('/content/drive/MyDrive/Diploma/sae_runs/k56')
SAE_CKPT_PATH = SAE_OUT_DIR / 'checkpoints' / 'sae_epoch_025.pt'
EVAL_OUT_DIR = Path('/content/drive/MyDrive/Diploma/sae_eval_k56_refusal_features')
FEATURE_OUT_DIR = EVAL_OUT_DIR / 'refusal_features_strict_retain005'
SELECTED_FEATURES_JSON_PATH = FEATURE_OUT_DIR / 'selected_refusal_features_strict.json'
TINY_OUT_DIR = EVAL_OUT_DIR / 'tinybenchmarks_input_formatted_top4'
TINY_OUT_DIR.mkdir(parents=True, exist_ok=True)

D_IN = 4096
D_SAE = 16384
K = 56
HOOK_LAYER = 12
TARGET_FEATURE_TOP_N = 4
CLAMP_VALUE = 0.0
MAX_INPUT_LENGTH = 2048
MAX_NEW_TOKENS_GSM8K = 512
DO_SAMPLE = False
GEN_USE_CACHE = True
VARIANTS = ['base', 'sae_intervention']
SAE_DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
MODEL_DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
LETTERS = [chr(ord('A') + i) for i in range(26)]

TINY_TASKS = {
    'tinyMMLU': {'hf_name': 'tinyBenchmarks/tinyMMLU', 'config': 'all', 'split': 'test', 'benchmark': 'mmlu', 'kind': 'mc'},
    'tinyHellaswag': {'hf_name': 'tinyBenchmarks/tinyHellaswag', 'config': None, 'split': 'validation', 'benchmark': 'hellaswag', 'kind': 'mc'},
    'tinyArc': {'hf_name': 'tinyBenchmarks/tinyAI2_arc', 'config': None, 'split': 'test', 'benchmark': 'arc', 'kind': 'mc'},
    'tinyTruthfulQA': {'hf_name': 'tinyBenchmarks/tinyTruthfulQA', 'config': 'multiple_choice', 'split': 'validation', 'benchmark': 'truthfulqa', 'kind': 'mc'},
    'tinyWinogrande': {'hf_name': 'tinyBenchmarks/tinyWinogrande', 'config': 'winogrande_xl', 'split': 'validation', 'benchmark': 'winogrande', 'kind': 'mc'},
    'tinyGSM8k': {'hf_name': 'tinyBenchmarks/tinyGSM8k', 'config': 'main', 'split': 'test', 'benchmark': 'gsm8k', 'kind': 'gen'},
}

In [ ]:
def read_json(path: Path):
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def write_json(path: Path, obj) -> None:
    with path.open('w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def choice_label(i: int) -> str:
    return LETTERS[i] if i < len(LETTERS) else f'choice_{i}'


def get_hidden_from_output(output):
    return output[0] if isinstance(output, tuple) else output


def replace_hidden_in_output(output, new_hidden):
    if isinstance(output, tuple):
        output = list(output)
        output[0] = new_hidden
        return tuple(output)
    return new_hidden


class TopKSAE(nn.Module):
    def __init__(self, d_in: int, d_sae: int, k: int, dtype: torch.dtype):
        super().__init__()
        self.W_enc = nn.Parameter(torch.empty(d_in, d_sae, dtype=dtype))
        self.W_dec = nn.Parameter(torch.empty(d_sae, d_in, dtype=dtype))
        self.b_enc = nn.Parameter(torch.zeros(d_sae, dtype=dtype))
        self.b_dec = nn.Parameter(torch.zeros(d_in, dtype=dtype))
        self.k = k
        self.d_sae = d_sae

    def encode_topk(self, x: torch.Tensor):
        pre_acts = (x - self.b_dec) @ self.W_enc + self.b_enc
        topk_vals, topk_idx = pre_acts.topk(self.k, dim=-1)
        return F.relu(topk_vals), topk_idx

    def encode_dense(self, x: torch.Tensor):
        topk_vals, topk_idx = self.encode_topk(x)
        feat = torch.zeros(x.shape[0], self.d_sae, device=x.device, dtype=x.dtype)
        feat.scatter_(-1, topk_idx, topk_vals)
        return feat

In [ ]:
sae = TopKSAE(D_IN, D_SAE, K, SAE_DTYPE).to(DEVICE)
sae.load_state_dict(torch.load(SAE_CKPT_PATH, map_location=DEVICE, weights_only=True)['model_state_dict'])
sae.eval()
TARGET_FEATURES = [int(x) for x in read_json(SELECTED_FEATURES_JSON_PATH)['selected_features'][:TARGET_FEATURE_TOP_N]]

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=MODEL_DTYPE, device_map='auto', low_cpu_mem_usage=True, attn_implementation='sdpa')
model.eval()
INPUT_DEVICE = next(model.model.embed_tokens.parameters()).device

In [ ]:
@torch.inference_mode()
def apply_delta_intervention_to_flat(x_flat: torch.Tensor) -> torch.Tensor:
    feat = sae.encode_dense(x_flat)
    delta = torch.zeros_like(x_flat)
    for j in TARGET_FEATURES:
        active = feat[:, j] > 0
        if active.any():
            delta[active] += (CLAMP_VALUE - feat[active, j]).unsqueeze(-1) * sae.W_dec[j].unsqueeze(0)
    return x_flat + delta


def patch_hidden(hidden: torch.Tensor, variant: str) -> torch.Tensor:
    if variant == 'base':
        return hidden
    b, s, d = hidden.shape
    x_flat = hidden.reshape(-1, d).to(DEVICE, dtype=SAE_DTYPE)
    patched = apply_delta_intervention_to_flat(x_flat)
    return patched.reshape(b, s, d).to(hidden.device, dtype=hidden.dtype)


@contextmanager
def patch_context(variant: str):
    if variant == 'base':
        yield
        return
    def hook(module, inputs, output):
        hidden = get_hidden_from_output(output)
        return replace_hidden_in_output(output, patch_hidden(hidden, variant))
    handle = model.model.layers[HOOK_LAYER].register_forward_hook(hook)
    try:
        yield
    finally:
        handle.remove()


def variant_context(variant: str):
    return nullcontext() if variant == 'base' else patch_context(variant)


def load_tiny_dataset(task_name: str) -> list[dict]:
    cfg = TINY_TASKS[task_name]
    ds = load_dataset(cfg['hf_name'], cfg['config'], split=cfg['split']) if cfg['config'] else load_dataset(cfg['hf_name'], split=cfg['split'])
    return [dict(x) for x in ds]


def get_prompt_from_row(row: dict) -> str:
    return str(row['input_formatted']).strip()


def normalize_answer_index(answer, choices=None):
    if answer is None:
        return None
    if isinstance(answer, (int, np.integer)):
        return int(answer)
    s = str(answer).strip()
    if s.isdigit():
        return int(s)
    if s.upper() in LETTERS:
        return LETTERS.index(s.upper())
    if choices is not None:
        for i, c in enumerate(choices):
            if str(c).strip().lower() == s.lower():
                return i
    return None


def get_first_existing(row: dict, names: list[str], default=None):
    for name in names:
        if name in row and row[name] is not None:
            return row[name]
    return default


def extract_mc_gold_and_choices(task_name: str, row: dict):
    if task_name == 'tinyTruthfulQA':
        targets = row['mc1_targets']
        choices = list(targets['choices'])
        return list(targets['labels']).index(1), choices
    if task_name == 'tinyWinogrande':
        return int(str(row['answer']).strip()) - 1, [str(row['option1']), str(row['option2'])]
    choices = get_first_existing(row, ['choices', 'options', 'endings'], [])
    if isinstance(choices, dict):
        if 'text' in choices:
            texts = list(choices['text'])
            labels = list(choices.get('label', []))
            answer = get_first_existing(row, ['answerKey', 'answer', 'target', 'label'])
            if isinstance(answer, str) and answer in labels:
                return labels.index(answer), texts
            return normalize_answer_index(answer, texts), texts
        choices = list(choices.values())
    answer = get_first_existing(row, ['answerKey', 'answer', 'target', 'label'])
    return normalize_answer_index(answer, list(choices)), list(choices)


def get_mc_prompt_and_continuations(task_name: str, row: dict):
    if task_name == 'tinyWinogrande':
        answer_idx, choices = extract_mc_gold_and_choices(task_name, row)
        prefix, suffix = str(row['sentence']).split('_', 1)
        return prefix, [choices[0] + suffix, choices[1] + suffix], choices, answer_idx
    answer_idx, choices = extract_mc_gold_and_choices(task_name, row)
    continuations = [LETTERS[i] for i in range(len(choices))] if task_name == 'tinyMMLU' else [str(c) for c in choices]
    return get_prompt_from_row(row), continuations, choices, answer_idx


def candidate_continuations(prompt_text: str, continuation: str) -> list[str]:
    return [str(continuation)] if prompt_text.endswith((' ', '\n')) else [' ' + str(continuation), str(continuation)]


@torch.inference_mode()
def continuation_logprob(prompt_text: str, continuation: str, variant: str) -> tuple[float, int]:
    prompt_ids = tokenizer(prompt_text, return_tensors='pt', add_special_tokens=False, truncation=True, max_length=MAX_INPUT_LENGTH)['input_ids']
    cont_ids = tokenizer(continuation, return_tensors='pt', add_special_tokens=False)['input_ids']
    input_ids = torch.cat([prompt_ids, cont_ids], dim=1).to(INPUT_DEVICE)
    attention_mask = torch.ones_like(input_ids, device=INPUT_DEVICE)
    prompt_len = prompt_ids.shape[1]
    cont_len = cont_ids.shape[1]
    with variant_context(variant):
        out = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)
    logits = out.logits[:, :-1, :]
    labels = input_ids[:, 1:]
    start = prompt_len - 1
    token_log_probs = torch.log_softmax(logits[:, start:start + cont_len, :], dim=-1).gather(-1, labels[:, start:start + cont_len].unsqueeze(-1)).squeeze(-1)
    return float(token_log_probs.sum().item()), int(cont_len)


@torch.inference_mode()
def predict_mc(row: dict, task_name: str, variant: str):
    prompt_text, continuations, choices, answer_idx = get_mc_prompt_and_continuations(task_name, row)
    raw_scores, norm_scores, chosen = {}, {}, {}
    for i, cont in enumerate(continuations):
        label = choice_label(i)
        scored = []
        for candidate in candidate_continuations(prompt_text, cont):
            score, n_tokens = continuation_logprob(prompt_text, candidate, variant)
            scored.append((score, score / max(n_tokens, 1), candidate))
        best = max(scored, key=lambda x: x[0] if task_name == 'tinyMMLU' else x[1])
        raw_scores[label], norm_scores[label], chosen[label] = best
    scores = raw_scores if task_name == 'tinyMMLU' else norm_scores
    pred_idx = int(np.argmax([scores[choice_label(i)] for i in range(len(continuations))]))
    return pred_idx, raw_scores, norm_scores, chosen, prompt_text, choices, answer_idx


def extract_last_number(text: str):
    matches = re.findall(r'[-+]?\d*\.?\d+', str(text).replace(',', ''))
    return matches[-1] if matches else None


def normalize_number(x):
    if x is None:
        return None
    v = float(str(x))
    return str(int(round(v))) if abs(v - round(v)) < 1e-6 else str(v)


def extract_gsm8k_gold(row: dict):
    answer = str(row.get('answer', row.get('target', row.get('label', ''))))
    return normalize_number(extract_last_number(answer.split('####')[-1] if '####' in answer else answer))


@torch.inference_mode()
def generate_completion(prompt_text: str, variant: str, max_new_tokens: int) -> str:
    batch = tokenizer(prompt_text, return_tensors='pt', truncation=True, max_length=MAX_INPUT_LENGTH, add_special_tokens=False)
    batch = {k: v.to(INPUT_DEVICE) for k, v in batch.items()}
    with variant_context(variant):
        output_ids = model.generate(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
            max_new_tokens=max_new_tokens,
            do_sample=DO_SAMPLE,
            use_cache=GEN_USE_CACHE,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output_ids[0, batch['input_ids'].shape[1]:], skip_special_tokens=True).strip()


def summarize_task_df(df: pd.DataFrame, benchmark: str, task_name: str, variant: str) -> dict:
    if 'manual_correct' in df.columns:
        mask = df['manual_correct'].notna()
        df.loc[mask, 'correct'] = df.loc[mask, 'manual_correct'].astype(int)
    df = df.sort_values('doc_id').copy()
    estimates = tb.evaluate(df['correct'].astype(float).to_numpy(), benchmark)
    est = estimates[benchmark] if benchmark in estimates else estimates
    return {'task': task_name, 'benchmark': benchmark, 'variant': variant, 'n': int(len(df)), 'raw_acc': float(df['correct'].mean()), 'irt': float(est['irt']), 'pirt': float(est['pirt']), 'gpirt': float(est['gpirt'])}


def evaluate_mc_task(task_name: str, variant: str):
    cfg = TINY_TASKS[task_name]
    rows_out = []
    for doc_id, row in enumerate(tqdm(load_tiny_dataset(task_name), desc=f'{task_name} | {variant}')):
        pred_idx, raw_scores, norm_scores, chosen, prompt, choices, answer_idx = predict_mc(row, task_name, variant)
        rows_out.append({
            'task': task_name,
            'benchmark': cfg['benchmark'],
            'variant': variant,
            'doc_id': doc_id,
            'prompt_input_formatted': prompt,
            'question': row.get('question', row.get('ctx', row.get('sentence'))),
            'choices': json.dumps(choices, ensure_ascii=False),
            'answer_idx': answer_idx,
            'answer_letter': choice_label(answer_idx) if answer_idx is not None else None,
            'prediction_idx': pred_idx,
            'prediction_letter': choice_label(pred_idx),
            'correct': int(pred_idx == answer_idx) if answer_idx is not None else 0,
            'raw_logprobs': json.dumps(raw_scores, ensure_ascii=False),
            'norm_logprobs': json.dumps(norm_scores, ensure_ascii=False),
            'chosen_continuations': json.dumps(chosen, ensure_ascii=False),
        })
    df = pd.DataFrame(rows_out)
    return df, summarize_task_df(df, cfg['benchmark'], task_name, variant)


def evaluate_gsm8k_task(task_name: str, variant: str):
    cfg = TINY_TASKS[task_name]
    rows_out = []
    for doc_id, row in enumerate(tqdm(load_tiny_dataset(task_name), desc=f'{task_name} | {variant}')):
        prompt = get_prompt_from_row(row)
        completion = generate_completion(prompt, variant, MAX_NEW_TOKENS_GSM8K)
        gold = extract_gsm8k_gold(row)
        pred = normalize_number(extract_last_number(completion))
        rows_out.append({
            'task': task_name,
            'benchmark': cfg['benchmark'],
            'variant': variant,
            'doc_id': doc_id,
            'prompt_input_formatted': prompt,
            'question': row.get('question'),
            'gold_answer': row.get('answer'),
            'gold_number': gold,
            'raw_completion': completion,
            'prediction_number': pred,
            'correct': int(pred is not None and gold is not None and pred == gold),
        })
    df = pd.DataFrame(rows_out)
    return df, summarize_task_df(df, cfg['benchmark'], task_name, variant)


def evaluate_task(task_name: str, variant: str):
    return evaluate_mc_task(task_name, variant) if TINY_TASKS[task_name]['kind'] == 'mc' else evaluate_gsm8k_task(task_name, variant)


def build_comparison(summary_df: pd.DataFrame) -> pd.DataFrame:
    base = summary_df[summary_df['variant'] == 'base']
    inter = summary_df[summary_df['variant'] == 'sae_intervention']
    out = base.merge(inter, on=['task', 'benchmark'], suffixes=('_base', '_intervention'))
    for metric in ['raw_acc', 'irt', 'pirt', 'gpirt']:
        out[f'delta_{metric}'] = out[f'{metric}_intervention'] - out[f'{metric}_base']
    return out[['task', 'benchmark', 'n_base', 'raw_acc_base', 'raw_acc_intervention', 'delta_raw_acc', 'gpirt_base', 'gpirt_intervention', 'delta_gpirt']]

In [ ]:
summaries = []
for task_name in TINY_TASKS:
    for variant in VARIANTS:
        rows_path = TINY_OUT_DIR / f'{task_name}_{variant}_rows.csv'
        summary_path = TINY_OUT_DIR / f'{task_name}_{variant}_summary.json'
        if rows_path.exists():
            df = pd.read_csv(rows_path)
            summary = summarize_task_df(df, TINY_TASKS[task_name]['benchmark'], task_name, variant)
        else:
            df, summary = evaluate_task(task_name, variant)
            df.to_csv(rows_path, index=False)
        write_json(summary_path, summary)
        summaries.append(summary)

summary_df = pd.DataFrame(summaries)
comparison_df = build_comparison(summary_df)
summary_df.to_csv(TINY_OUT_DIR / 'tinybenchmarks_summary_all.csv', index=False)
comparison_df.to_csv(TINY_OUT_DIR / 'tinybenchmarks_base_vs_intervention.csv', index=False)
display(summary_df)
display(comparison_df)